In [2]:
%pip install opencv-python numpy ultralytics

Note: you may need to restart the kernel to use updated packages.Collecting opencv-python
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/44.0 MB 389.8 kB/s eta 0:01:52
   ---------------------------------------- 0.5/44.0 MB 389.8 kB/s eta 0:01:52
   ---------------------------------------- 0.5/44.0 MB 389.8 kB/s eta 0:01:52
    --------------------------------------- 0.8/44.0 MB 398.9 kB/s eta 0:01:49
    --------------------------------------- 0.8/44.0 MB 398.9 kB/s eta 0:01:49
    --------------------------------------- 0.8/44.0 MB 398.9 kB/s eta 0:01:49
    ------------

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO

print("OpenCV version:", cv2.__version__)
print("All libraries installed successfully!")

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\dell\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
OpenCV version: 5.0.0
All libraries installed successfully!


In [1]:

import cv2
import numpy as np
from ultralytics import YOLO

# Load YOLO pose estimation model
model = YOLO("yolo26n-pose.pt")

# Read input video
cap = cv2.VideoCapture("sample.mp4")

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Create output video
out = cv2.VideoWriter(
    "coco_points_output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Run pose estimation
    results = model(frame, conf=0.25, verbose=False)

    # Create a black background
    output_frame = np.zeros((height, width, 3), dtype=np.uint8)

    # Draw only COCO keypoints
    if results[0].keypoints is not None:
        keypoints = results[0].keypoints.xy.cpu().numpy()
        confidences = results[0].keypoints.conf

        if confidences is not None:
            confidences = confidences.cpu().numpy()

        for person_idx, person_points in enumerate(keypoints):
            for point_idx, (x, y) in enumerate(person_points):
                # Ignore missing or low-confidence keypoints
                if x == 0 and y == 0:
                    continue

                if confidences is not None:
                    if confidences[person_idx][point_idx] < 0.25:
                        continue

                # Draw a small point
                cv2.circle(
                    output_frame,
                    (int(x), int(y)),
                    5,
                    (0, 255, 0),
                    -1
                )

    # Save output frame
    out.write(output_frame)

    # Display only the COCO points
    cv2.imshow("COCO Keypoints Only", output_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Done! Output saved as coco_points_output.mp4")


Done! Output saved as coco_points_output.mp4
